# 02 — Clasificación, Pre-OCR y Segmentación de Cuadrantes (Agente 1)

**Objetivo:** Validar el pipeline de visión por computadora del **Agente 1**:
1. Renderizado en alta definición (300 DPI) y Pre-OCR de cabecera.
2. Detección de layout (`clasificacion_4`, `clasificacion_2`, `eliminatoria`, `portada`) y descarte de cuadrantes sin tinta.
3. Visualización de recortes generados y comparación antes/después del filtro de mejora de calidad (`ImageEnhancer`).

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../src'))

import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

try:
    from tiro_arco.agente_1_clasificador_segmentador import AgenteClasificadorSegmentador
except ImportError:
    from agente_1_clasificador_segmentador_2 import AgenteClasificadorSegmentador

RAW_DIR = '../data/raw'
muestra = os.path.join(RAW_DIR, '1-AFPSF_Plani_TS-4-Arqueros.png')
if not os.path.exists(muestra):
    candidatos = [os.path.join(RAW_DIR, f) for f in os.listdir(RAW_DIR) if f.lower().endswith(('.png', '.jpg', '.pdf'))]
    muestra = sorted(candidatos)[0]

agente1 = AgenteClasificadorSegmentador(output_dir='/tmp/nb02_agent1_crops')
manifiesto_1 = agente1.procesar_documento(muestra)

print(f'Archivo procesado: {manifiesto_1["archivo_origen"]} | Páginas: {manifiesto_1["total_paginas"]}')
df_tareas = pd.DataFrame(manifiesto_1['tareas_agente2'])
display(df_tareas[['pagina', 'target_id', 'label', 'tipo_juego', 'layout', 'es_pagina_informacion', 'es_cuadrante_vacio']])

In [ ]:
tareas = manifiesto_1.get('tareas_agente2', [])
n = len(tareas)
cols = min(2, max(1, n))
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = [axes] if n == 1 else axes.flatten()

for idx, t in enumerate(tareas):
    img = Image.open(t['ruta_recorte'])
    axes[idx].imshow(img)
    estado = 'VACÍO' if t['es_cuadrante_vacio'] else ('INFO/EXCLUIDO' if t['es_pagina_informacion'] else 'ACTIVO')
    axes[idx].set_title(f"{t['label']} | {t['layout']} [{estado}]", fontsize=10)
    axes[idx].axis('off')

for j in range(idx + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
if tareas:
    lbl_prueba = tareas[0]['label']
    ruta_orig = tareas[0]['ruta_recorte_original']
    ok, msg = agente1.mejorar_calidad_recorte(lbl_prueba)
    ruta_mej = tareas[0]['ruta_recorte']
    print(msg)

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(Image.open(ruta_orig))
    ax[0].set_title('Recorte Original (300 DPI)')
    ax[0].axis('off')

    ax[1].imshow(Image.open(ruta_mej))
    ax[1].set_title('Recorte Mejorado (CLAHE + Bilateral + Sharpen)')
    ax[1].axis('off')
    plt.tight_layout()
    plt.show()